# 10. 지하철 관련 데이터 전처리

서울교통공사 역별 월별 장애인 승하차인원과 교통약자 이용시설 API 원본 CSV를 함께 전처리한다. 장애인콜택시 정제 데이터의 `출발구`·`출발동` 또는 `목적구`·`목적동`에 등장한 구·동에 위치한 지하철역만 선별하고, 해당 역의 접근성 시설 정보를 결합해 장애인 네비게이터 분석용 데이터를 만든다.

이 노트북은 별도 매핑 CSV나 외부 파이썬 전처리 파일을 만들지 않고, 노트북 안에서 원본 로딩, ETL 연결, 결측치 처리, 이상치 검증, 접근성 시설 집계, 정제, 저장까지 수행한다.

## 0. 환경 설정

### 0-1. 라이브러리 로드

In [ ]:
from io import BytesIO
from pathlib import Path
import os
import json
import time
import unicodedata

os.environ.setdefault('MPLCONFIGDIR', '/tmp/calltaxi-da-matplotlib')

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from matplotlib.path import Path as MplPath

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

### 0-2. 프로젝트 경로와 입출력 파일 설정

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'


def normalize_filename(name: str) -> str:
    """macOS 한글 파일명(NFD)과 일반 한글 파일명(NFC)을 같은 이름처럼 비교한다."""
    return unicodedata.normalize('NFC', name).strip()


def resolve_existing_file(directory: Path, expected_name: str, keywords: tuple[str, ...] = ()) -> Path:
    """예상 파일명이 정확히 안 맞아도 같은 원본 파일을 찾아 반환한다."""
    direct_path = directory / expected_name
    if direct_path.exists():
        return direct_path

    expected_normalized = normalize_filename(expected_name)
    for candidate in directory.iterdir():
        if normalize_filename(candidate.name) == expected_normalized:
            return candidate

    matched = []
    for candidate in directory.iterdir():
        candidate_normalized = normalize_filename(candidate.name)
        if all(keyword in candidate_normalized for keyword in keywords):
            matched.append(candidate)

    if len(matched) == 1:
        return matched[0]
    if len(matched) > 1:
        names = [path.name for path in matched]
        raise FileNotFoundError(f'입력 파일 후보가 여러 개입니다. 하나만 남겨주세요: {names}')

    available = [path.name for path in directory.iterdir()]
    raise FileNotFoundError(
        f'입력 파일을 찾지 못했습니다. expected={expected_name}, keywords={keywords}, available={available}'
    )


RIDERSHIP_RAW_PATH = resolve_existing_file(
    RAW_DIR,
    '서울교통공사_역별 월별 장애인 승하차인원 정보_20251231.csv',
    keywords=('서울교통공사', '역별', '월별', '장애인', '승하차인원'),
)
TAXI_PATH = PROCESSED_DIR / '서울시설공단_장애인콜택시 탑승내역_정제_20251231.csv'
OUTPUT_PATH = PROCESSED_DIR / '서울교통공사_장애인_지하철_승하차인원_정제_20251231.csv'

ACCESSIBILITY_FILES = {
    '엘리베이터': RAW_DIR / '교통약자이용정보_엘리베이터.csv',
    '장애인화장실': RAW_DIR / '교통약자이용정보_장애인화장실.csv',
    '휠체어리프트': RAW_DIR / '교통약자이용정보_휠체어리프트.csv',
    '휠체어급속충전기': RAW_DIR / '교통약자이용정보_휠체어급속충전기.csv',
    '안전발판': RAW_DIR / '교통약자이용정보_안전발판보유현황.csv',
}
EXCLUDED_ACCESSIBILITY_FILES = {
    '에스컬레이터': '휠체어 이동 경로 판단과의 직접성이 낮아 제외',
    '무빙워크': '데이터 규모가 작고 휠체어 이동 핵심 지표가 아니어서 제외',
    '수어영상전화기': '역사 내 의사소통 지원 정보로, 이동 가능성 판단 지표에서는 제외',
    '교통약자도우미현황': '데이터 범위가 제한적이어서 1차 정제에서는 제외',
}

print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'장애인 승하차 원본: {RIDERSHIP_RAW_PATH}')
print(f'콜택시 정제 파일: {TAXI_PATH}')
print(f'저장 파일: {OUTPUT_PATH}')

### 0-3. 입력 파일 존재 확인

In [ ]:
assert RIDERSHIP_RAW_PATH.exists(), f'장애인 승하차 원본 파일이 없습니다: {RIDERSHIP_RAW_PATH}'
assert TAXI_PATH.exists(), f'콜택시 정제 파일이 없습니다: {TAXI_PATH}'
for name, file_path in ACCESSIBILITY_FILES.items():
    assert file_path.exists(), f'{name} 원본 파일이 없습니다: {file_path}'
print('입력 파일 확인 완료')
print(f'사용 접근성 데이터: {list(ACCESSIBILITY_FILES)}')
print(f'제외 접근성 데이터: {list(EXCLUDED_ACCESSIBILITY_FILES)}')

### 0-4. 원본 데이터 출처

원본 CSV와 노트북 실행 중 불러오는 외부 데이터의 출처를 명시한다. API 인증키는 `.env`에서만 관리하고, 노트북에는 공개 가능한 base URL과 상세기능 경로만 남긴다.

In [ ]:
MOBILITY_API_BASE_URL = 'https://apis.data.go.kr/B553766/wksn'

DATA_SOURCE_INFO = [
    {
        '구분': '장애인 지하철 이용량',
        '사용 파일': RIDERSHIP_RAW_PATH.name,
        '출처명': '서울교통공사_역별 월별 장애인 승하차인원 정보',
        '출처 URL 또는 API': 'https://data.seoul.go.kr/dataList/OA-22478/L/1/datasetView.do',
        '비고': '1~8호선 역별·월별 장애인 승차/하차 인원 원본 CSV',
    },
    {
        '구분': '콜택시 지역 기준',
        '사용 파일': TAXI_PATH.name,
        '출처명': '서울시설공단_장애인콜택시 탑승내역',
        '출처 URL 또는 API': 'https://www.data.go.kr/data/15115859/fileData.do',
        '비고': '이미 정제된 processed CSV를 사용해 출발구·출발동/목적구·목적동 기준 생성',
    },
    {
        '구분': '교통약자 시설',
        '사용 파일': ACCESSIBILITY_FILES['엘리베이터'].name,
        '출처명': '교통약자이용정보_엘리베이터 조회',
        '출처 URL 또는 API': f'{MOBILITY_API_BASE_URL}/getWksnElvtr',
        '비고': '장애인 네비게이터 핵심 시설로 사용',
    },
    {
        '구분': '교통약자 시설',
        '사용 파일': ACCESSIBILITY_FILES['장애인화장실'].name,
        '출처명': '교통약자이용정보_장애인화장실 조회',
        '출처 URL 또는 API': f'{MOBILITY_API_BASE_URL}/getWksnRstrm',
        '비고': '보조 접근성 시설로 사용',
    },
    {
        '구분': '교통약자 시설',
        '사용 파일': ACCESSIBILITY_FILES['휠체어리프트'].name,
        '출처명': '교통약자이용정보_휠체어리프트 조회',
        '출처 URL 또는 API': f'{MOBILITY_API_BASE_URL}/getWksnWhcllift',
        '비고': '엘리베이터 부재/대체 이동수단 판단에 사용',
    },
    {
        '구분': '교통약자 시설',
        '사용 파일': ACCESSIBILITY_FILES['휠체어급속충전기'].name,
        '출처명': '교통약자이용정보_휠체어급속충전기 조회',
        '출처 URL 또는 API': f'{MOBILITY_API_BASE_URL}/getWksnWhclCharge',
        '비고': '전동휠체어 이용자 편의시설로 사용',
    },
    {
        '구분': '교통약자 시설',
        '사용 파일': ACCESSIBILITY_FILES['안전발판'].name,
        '출처명': '교통약자이용정보_안전발판보유현황 조회',
        '출처 URL 또는 API': f'{MOBILITY_API_BASE_URL}/getWksnSafePlfm',
        '비고': '승강장 탑승 안전성 보조 지표로 사용',
    },
    {
        '구분': '행정동 경계',
        '사용 파일': '노트북 실행 중 메모리 로드',
        '출처명': 'HangJeongDong 행정동 경계 GeoJSON',
        '출처 URL 또는 API': 'https://raw.githubusercontent.com/vuski/admdongkor/master/ver20250701/HangJeongDong_ver20250701.geojson',
        '비고': '파일 저장 없이 지하철역의 행정동 매핑에 사용',
    },
    {
        '구분': '지하철역 좌표',
        '사용 파일': '노트북 실행 중 메모리 로드',
        '출처명': '서울시 역코드로 지하철역 정보 검색 CSV',
        '출처 URL 또는 API': 'https://raw.githubusercontent.com/hayoungishere/KMA/master/data/%EC%84%9C%EC%9A%B8%EC%8B%9C%20%EC%97%AD%EC%BD%94%EB%93%9C%EB%A1%9C%20%EC%A7%80%ED%95%98%EC%B2%A0%EC%97%AD%20%EC%A0%95%EB%B3%B4%20%EA%B2%80%EC%83%89.csv',
        '비고': '파일 저장 없이 위경도 기반 행정동 매핑에 사용',
    },
]

source_info = pd.DataFrame(DATA_SOURCE_INFO)
display(source_info)

## 1. 데이터 수집 및 원본 로딩

### 1-1. 입력 컬럼과 인코딩 설정

In [ ]:
RAW_REQUIRED_COLUMNS = ['연번', '호선', '고유역번호(외부역코드)', '역명', '승하차구분', '수송연월', '승하차인원수']
ENCODINGS = ('utf-8-sig', 'utf-8', 'cp949', 'euc-kr')
MONTH_MAP = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12,
}

def read_csv_with_encoding(file_path, required_columns=None):
    last_error = None
    for encoding in ENCODINGS:
        try:
            frame = pd.read_csv(file_path, encoding=encoding, index_col=False)
            if required_columns is not None:
                missing_columns = set(required_columns) - set(frame.columns)
                assert not missing_columns, f'{file_path.name} 누락 컬럼: {sorted(missing_columns)}'
                frame = frame[required_columns].copy()
            return frame, encoding
        except UnicodeDecodeError as error:
            last_error = error
    raise UnicodeError(f'{file_path.name} 인코딩을 확인할 수 없습니다') from last_error

def parse_transport_month(value):
    value = str(value).strip()
    month_abbr, year_suffix = value.split('-')
    year = 2000 + int(year_suffix)
    month = MONTH_MAP[month_abbr]
    return pd.Timestamp(year=year, month=month, day=1)

### 1-2. 장애인 승하차 원본 CSV 로드

In [ ]:
started_at = time.perf_counter()
ridership_raw, ridership_encoding = read_csv_with_encoding(RIDERSHIP_RAW_PATH, RAW_REQUIRED_COLUMNS)
print(f'원본 인코딩: {ridership_encoding}')
print(f'원본 행 수: {len(ridership_raw):,}행')
display(ridership_raw.head())

### 1-3. 원본 구조 확인

In [ ]:
raw_profile = pd.Series({
    '행 수': len(ridership_raw),
    '컬럼 수': ridership_raw.shape[1],
    '호선 수': ridership_raw['호선'].nunique(),
    '역 수': ridership_raw['역명'].nunique(),
    '수송연월 수': ridership_raw['수송연월'].nunique(),
    '승하차구분 수': ridership_raw['승하차구분'].nunique(),
})
display(raw_profile.to_frame('값'))
display(ridership_raw['승하차구분'].value_counts().to_frame('행 수'))
display(ridership_raw['수송연월'].drop_duplicates().to_frame('수송연월').reset_index(drop=True))

## 2. ETL 변환 및 품질 검증

### 2-1. 자료형 변환과 표준 컬럼 생성

In [ ]:
ridership = ridership_raw.copy()

for column in ['역명', '승하차구분', '수송연월']:
    ridership[column] = ridership[column].astype('string').str.strip()

ridership['호선'] = pd.to_numeric(ridership['호선'], errors='coerce').astype('Int64')
ridership['고유역번호(외부역코드)'] = pd.to_numeric(ridership['고유역번호(외부역코드)'], errors='coerce').astype('Int64')
ridership['승하차인원수'] = pd.to_numeric(ridership['승하차인원수'], errors='coerce')
ridership['사용월'] = ridership['수송연월'].map(parse_transport_month)
ridership['노선명'] = ridership['호선'].astype('Int64').astype('string') + '호선'
ridership['역명_정규화'] = ridership['역명'].str.replace(r'역$', '', regex=True).str.replace(r'\([^)]*\)', '', regex=True).str.strip()
ridership['역명_정규화'] = ridership['역명_정규화'].replace({'서울역': '서울'})

ridership.head()

### 2-2. 결측치 처리 및 필수값 검증

In [ ]:
required_after_transform = ['호선', '고유역번호(외부역코드)', '역명', '승하차구분', '수송연월', '승하차인원수', '사용월', '노선명']
missing_counts = ridership[required_after_transform].isna().sum()
assert missing_counts.sum() == 0, f'결측 또는 변환 실패: {missing_counts[missing_counts.gt(0)].to_dict()}'
print('필수 컬럼 결측 없음')
display(missing_counts.to_frame('결측 수'))

### 2-3. 날짜·범주·음수 이상치 검증

In [ ]:
expected_months = pd.date_range('2025-01-01', '2025-12-01', freq='MS')
actual_months = pd.DatetimeIndex(ridership['사용월'].drop_duplicates()).sort_values()
assert actual_months.equals(expected_months), f'누락 월: {expected_months.difference(actual_months).tolist()}'
assert set(ridership['승하차구분'].dropna()) == {'승차', '하차'}, '승하차구분은 승차/하차만 허용합니다.'
assert ridership['승하차인원수'].ge(0).all(), '음수 승하차 인원수가 있습니다.'
assert ridership['승하차인원수'].mod(1).eq(0).all(), '승하차인원수에 정수가 아닌 값이 있습니다.'
ridership['승하차인원수'] = ridership['승하차인원수'].astype('int64')
print('월 범위, 승하차구분, 음수 검증 완료')

### 2-4. 중복 검증과 승차·하차 컬럼 분리

In [ ]:
long_key = ['사용월', '노선명', '역명', '승하차구분']
long_duplicate_count = int(ridership.duplicated(long_key).sum())
assert long_duplicate_count == 0, f'월·노선·역·승하차구분 중복이 {long_duplicate_count:,}행 있습니다.'

ridership_wide = (
    ridership.pivot_table(
        index=['사용월', '수송연월', '노선명', '호선', '고유역번호(외부역코드)', '역명', '역명_정규화'],
        columns='승하차구분', values='승하차인원수', aggfunc='sum', fill_value=0
    )
    .reset_index()
    .rename(columns={'승차': '장애인승차인원수', '하차': '장애인하차인원수'})
)
ridership_wide.columns.name = None
ridership_wide['장애인총승하차인원수'] = ridership_wide['장애인승차인원수'] + ridership_wide['장애인하차인원수']

wide_key = ['사용월', '노선명', '역명']
wide_duplicate_count = int(ridership_wide.duplicated(wide_key).sum())
assert wide_duplicate_count == 0, f'월·노선·역 중복이 {wide_duplicate_count:,}행 있습니다.'

ridership_wide = ridership_wide.sort_values(['사용월', '호선', '역명']).reset_index(drop=True)
display(ridership_wide.head())

### 2-5. 통계적 이상치 확인

In [ ]:
station_stats = (
    ridership_wide.groupby(['노선명', '역명'])['장애인총승하차인원수']
    .agg(['mean', 'std'])
    .rename(columns={'mean': '역월평균', 'std': '역월표준편차'})
    .reset_index()
)
ridership_wide = ridership_wide.merge(station_stats, on=['노선명', '역명'], how='left')
ridership_wide['이상치여부'] = np.where(
    ridership_wide['역월표준편차'].fillna(0).eq(0),
    False,
    (ridership_wide['장애인총승하차인원수'] - ridership_wide['역월평균']).abs() > 3 * ridership_wide['역월표준편차']
)

outlier_count = int(ridership_wide['이상치여부'].sum())
print(f'통계적 이상치 후보: {outlier_count:,}행')
display(
    ridership_wide.loc[ridership_wide['이상치여부']]
    .sort_values('장애인총승하차인원수', ascending=False)
    .head(20)
)

# 월별 공공 통계는 특정 월 수요 급증 자체가 의미 있는 신호일 수 있어 자동 삭제하지 않는다.
# 음수·결측·중복처럼 명백한 오류만 제거/중단하고, 통계적 이상치 후보는 표시 컬럼으로 남긴다.

### 2-6. 품질 요약 확인

In [ ]:
quality_summary = pd.Series({
    '원본 행 수': len(ridership_raw),
    '정제 후 월별 역 행 수': len(ridership_wide),
    '월 수': ridership_wide['사용월'].nunique(),
    '노선 수': ridership_wide['노선명'].nunique(),
    '역 수': ridership_wide['역명'].nunique(),
    '중복 키 수': wide_duplicate_count,
    '결측값 수': int(ridership_wide.isna().sum().sum()),
    '총 장애인 승차 인원': int(ridership_wide['장애인승차인원수'].sum()),
    '총 장애인 하차 인원': int(ridership_wide['장애인하차인원수'].sum()),
    '통계적 이상치 후보 수': outlier_count,
})
display(quality_summary.to_frame('값'))

## 3. 콜택시 출발·목적 구·동 기준 생성

### 3-1. 콜택시 정제 데이터 로드

In [ ]:
taxi_regions = pd.read_csv(
    TAXI_PATH,
    encoding='utf-8-sig',
    usecols=['출발구', '출발동', '목적구', '목적동'],
)
for column in taxi_regions.columns:
    taxi_regions[column] = taxi_regions[column].astype('string').str.strip()

taxi_regions.head()

### 3-2. 출발·목적 구·동 조합 만들기

In [ ]:
def normalize_area_name(value):
    if pd.isna(value):
        return pd.NA
    return str(value).strip().replace(' ', '')

start_pairs = taxi_regions[['출발구', '출발동']].rename(columns={'출발구': '지역명', '출발동': '행정동명'})
end_pairs = taxi_regions[['목적구', '목적동']].rename(columns={'목적구': '지역명', '목적동': '행정동명'})
used_region_dongs = pd.concat([start_pairs, end_pairs], ignore_index=True).dropna().drop_duplicates()
used_region_dongs['지역명_정규화'] = used_region_dongs['지역명'].map(normalize_area_name)
used_region_dongs['행정동명_정규화'] = used_region_dongs['행정동명'].map(normalize_area_name)
used_region_dong_keys = set(zip(used_region_dongs['지역명_정규화'], used_region_dongs['행정동명_정규화']))
used_regions = set(used_region_dongs['지역명'].dropna())

print(f'콜택시 출발·목적 지역 수: {len(used_regions):,}개')
print(f'콜택시 구·동 조합 수: {len(used_region_dong_keys):,}개')
display(used_region_dongs.head(20))

## 4. 행정동 경계와 지하철역 좌표 매핑

### 4-1. 행정동 경계와 지하철역 좌표 로드

In [ ]:
# 파일로 저장하지 않고 메모리에서만 읽어, 별도 매핑 원본 파일을 남기지 않는다.
BOUNDARY_URL = 'https://raw.githubusercontent.com/vuski/admdongkor/master/ver20250701/HangJeongDong_ver20250701.geojson'
STATION_URL = 'https://raw.githubusercontent.com/hayoungishere/KMA/master/data/%EC%84%9C%EC%9A%B8%EC%8B%9C%20%EC%97%AD%EC%BD%94%EB%93%9C%EB%A1%9C%20%EC%A7%80%ED%95%98%EC%B2%A0%EC%97%AD%20%EC%A0%95%EB%B3%B4%20%EA%B2%80%EC%83%89.csv'

boundary_response = requests.get(BOUNDARY_URL, timeout=30)
boundary_response.raise_for_status()
boundary_geojson = boundary_response.json()

station_response = requests.get(STATION_URL, timeout=30)
station_response.raise_for_status()
station_coordinates = pd.read_csv(BytesIO(station_response.content), encoding='utf-8')

print(f'행정동 경계 수: {len(boundary_geojson["features"]):,}개')
print(f'지하철역 좌표 행 수: {len(station_coordinates):,}행')
display(station_coordinates.head())

### 4-2. 공간 매칭 함수 정의

In [ ]:
def geometry_contains_point(geometry, longitude, latitude):
    point = (longitude, latitude)

    def polygon_contains(rings):
        exterior = np.asarray(rings[0], dtype='float64')
        if not MplPath(exterior).contains_point(point):
            return False
        for hole in rings[1:]:
            if MplPath(np.asarray(hole, dtype='float64')).contains_point(point):
                return False
        return True

    if geometry['type'] == 'Polygon':
        return polygon_contains(geometry['coordinates'])
    if geometry['type'] == 'MultiPolygon':
        return any(polygon_contains(polygon) for polygon in geometry['coordinates'])
    return False


def station_name_key(value):
    return str(value).strip().replace('역', '').replace(' ', '')

### 4-3. 콜택시 구·동에 해당하는 행정동 경계 선별

In [ ]:
candidate_boundaries = []
for feature in boundary_geojson['features']:
    properties = feature['properties']
    region_name = properties.get('sggnm')
    dong_name = properties.get('adm_nm', '').split()[-1]
    key = (normalize_area_name(region_name), normalize_area_name(dong_name))
    if key in used_region_dong_keys:
        candidate_boundaries.append({
            '지역명': region_name,
            '행정동명': dong_name,
            'geometry': feature['geometry'],
        })

print(f'콜택시 구·동 후보 경계 수: {len(candidate_boundaries):,}개')
assert candidate_boundaries, '콜택시 구·동과 일치하는 행정동 경계를 찾지 못했습니다.'

candidate_boundary_table = pd.DataFrame([{k: v for k, v in row.items() if k != 'geometry'} for row in candidate_boundaries])
display(candidate_boundary_table.head(20))

### 4-4. 지하철역 좌표를 행정동에 매칭

In [ ]:
if '역명' not in station_coordinates.columns and '전철역명' in station_coordinates.columns:
    station_coordinates = station_coordinates.rename(columns={'전철역명': '역명'})

coordinate_required_columns = ['호선', '역명', 'X좌표(WGS)', 'Y좌표(WGS)']
missing_coordinate_columns = set(coordinate_required_columns) - set(station_coordinates.columns)
assert not missing_coordinate_columns, f'역 좌표 누락 컬럼: {sorted(missing_coordinate_columns)}'

station_coordinates = station_coordinates.dropna(subset=['X좌표(WGS)', 'Y좌표(WGS)']).copy()
station_coordinates['X좌표(WGS)'] = pd.to_numeric(station_coordinates['X좌표(WGS)'], errors='coerce')
station_coordinates['Y좌표(WGS)'] = pd.to_numeric(station_coordinates['Y좌표(WGS)'], errors='coerce')
station_coordinates = station_coordinates.dropna(subset=['X좌표(WGS)', 'Y좌표(WGS)'])
station_coordinates['노선코드'] = station_coordinates['호선'].astype('string').str.strip()
station_coordinates['역명_정규화'] = station_coordinates['역명'].map(station_name_key)

matched_stations = []
for _, station in station_coordinates.iterrows():
    # 좌표 파일은 X좌표(WGS)=위도, Y좌표(WGS)=경도 형태다.
    latitude = station['X좌표(WGS)']
    longitude = station['Y좌표(WGS)']
    for boundary in candidate_boundaries:
        if geometry_contains_point(boundary['geometry'], longitude, latitude):
            matched_stations.append({
                '노선코드': station['노선코드'],
                '역명_정규화': station['역명_정규화'],
                '지역명': boundary['지역명'],
                '행정동명': boundary['행정동명'],
            })
            break

station_region = pd.DataFrame(matched_stations).drop_duplicates()
station_region = station_region.drop_duplicates(['노선코드', '역명_정규화'])
assert not station_region.duplicated(['노선코드', '역명_정규화']).any(), '한 역이 여러 행정동에 중복 매핑되었습니다.'

print(f'콜택시 구·동 내 지하철역 매핑 수: {len(station_region):,}개')
display(station_region.head(20))

## 5. 교통약자 이용시설 데이터 전처리

### 5-1. 분석에 사용할 접근성 시설 원본 로드

In [ ]:
accessibility_raw = {}
accessibility_load_summary = []
for facility_name, file_path in ACCESSIBILITY_FILES.items():
    frame, encoding = read_csv_with_encoding(file_path)
    for column in frame.select_dtypes(include=['object', 'string']).columns:
        frame[column] = frame[column].astype('string').str.strip()
    accessibility_raw[facility_name] = frame
    accessibility_load_summary.append({
        '시설구분': facility_name,
        '파일명': file_path.name,
        '인코딩': encoding,
        '행 수': len(frame),
        '컬럼 수': frame.shape[1],
        '역 수': frame['stnNm'].nunique() if 'stnNm' in frame.columns else pd.NA,
    })

display(pd.DataFrame(accessibility_load_summary))

### 5-2. 접근성 시설별 역 단위 집계

In [ ]:
def prepare_facility_base(frame):
    required = {'lineNm', 'stnNm'}
    missing = required - set(frame.columns)
    assert not missing, f'접근성 시설 데이터 누락 컬럼: {sorted(missing)}'
    base = frame.copy()
    base['노선명'] = base['lineNm'].astype('string').str.strip()
    base['역명_정규화'] = base['stnNm'].astype('string').map(station_name_key)
    return base


def clean_text(value):
    if pd.isna(value):
        return ''
    text = str(value).strip()
    if text.lower() in {'nan', 'none', '<na>'}:
        return ''
    return text


def unique_join(values):
    cleaned = []
    for value in values:
        text = clean_text(value)
        if text and text not in cleaned:
            cleaned.append(text)
    return '; '.join(cleaned)


def floor_label(section, floor):
    section = clean_text(section)
    floor = clean_text(floor)
    if section and floor:
        return f'{section}{floor}'
    return floor or section


def make_elvtr_detail(row):
    start_floor = floor_label(row.get('bgngFlrGrndUdgdSe'), row.get('bgngFlr'))
    end_floor = floor_label(row.get('endFlrGrndUdgdSe'), row.get('endFlr'))
    route = f'{start_floor}→{end_floor}' if start_floor and end_floor else start_floor or end_floor
    parts = [clean_text(row.get('mngNo')), route, clean_text(row.get('dtlPstn'))]
    return ' | '.join([part for part in parts if part])


def make_lift_detail(row):
    start_floor = floor_label(row.get('bgngFlrGrndUdgdSe'), row.get('bgngFlr'))
    end_floor = floor_label(row.get('endFlrGrndUdgdSe'), row.get('endFlr'))
    route = f'{start_floor}→{end_floor}' if start_floor and end_floor else start_floor or end_floor
    start_position = clean_text(row.get('bgngFlrDtlPstn'))
    end_position = clean_text(row.get('endFlrDtlPstn'))
    position = f'{start_position}→{end_position}' if start_position and end_position and start_position != end_position else start_position or end_position
    parts = [clean_text(row.get('mngNo')), route, position]
    return ' | '.join([part for part in parts if part])


facility_summaries = []

elv = prepare_facility_base(accessibility_raw['엘리베이터'])
elv['엘리베이터상세'] = elv.apply(make_elvtr_detail, axis=1)
elv_station = elv.groupby(['노선명', '역명_정규화'], as_index=False).agg(
    엘리베이터수=('fcltNo', 'count'),
    운행엘리베이터수=('oprtngSitu', lambda s: int(s.astype('string').eq('M').sum())),
    엘리베이터연결층=('엘리베이터상세', unique_join),
    엘리베이터설치위치=('dtlPstn', unique_join),
)
facility_summaries.append(('엘리베이터', elv_station))

rstrm = prepare_facility_base(accessibility_raw['장애인화장실'])
rstrm_station = rstrm.groupby(['노선명', '역명_정규화'], as_index=False).agg(
    장애인화장실수=('fcltNo', 'count'),
    휠체어접근가능화장실수=('whlchrAcsPsbltyYn', lambda s: int(s.astype('string').eq('Y').sum())),
    장애인화장실설치위치=('dtlPstn', unique_join),
    장애인화장실층=('stnFlr', unique_join),
)
facility_summaries.append(('장애인화장실', rstrm_station))

lift = prepare_facility_base(accessibility_raw['휠체어리프트'])
lift['휠체어리프트상세'] = lift.apply(make_lift_detail, axis=1)
lift_station = lift.groupby(['노선명', '역명_정규화'], as_index=False).agg(
    휠체어리프트수=('fcltNo', 'count'),
    운행휠체어리프트수=('oprtngSitu', lambda s: int(s.astype('string').eq('M').sum())),
    휠체어리프트운행구간=('휠체어리프트상세', unique_join),
    휠체어리프트설치위치=('bgngFlrDtlPstn', unique_join),
)
facility_summaries.append(('휠체어리프트', lift_station))

charge = prepare_facility_base(accessibility_raw['휠체어급속충전기'])
charge_station = charge.groupby(['노선명', '역명_정규화'], as_index=False).agg(
    휠체어급속충전기수=('elctcFacCnt', lambda s: int(pd.to_numeric(s, errors='coerce').fillna(0).sum())),
    휠체어급속충전기설치위치=('dtlPstn', unique_join),
    휠체어급속충전기층=('stnFlr', unique_join),
)
facility_summaries.append(('휠체어급속충전기', charge_station))

safe = prepare_facility_base(accessibility_raw['안전발판'])
safe_station = safe.groupby(['노선명', '역명_정규화'], as_index=False).agg(
    안전발판수=('sftyScfldEn', lambda s: int(s.astype('string').eq('Y').sum())),
)
facility_summaries.append(('안전발판', safe_station))

display(pd.DataFrame([
    {'시설구분': name, '역-노선 조합 수': len(summary)}
    for name, summary in facility_summaries
]))

### 5-3. 접근성 시설 통합 테이블 생성

In [ ]:
accessibility_station = None
for _, summary in facility_summaries:
    if accessibility_station is None:
        accessibility_station = summary.copy()
    else:
        accessibility_station = accessibility_station.merge(summary, on=['노선명', '역명_정규화'], how='outer')

count_columns = [
    '엘리베이터수', '운행엘리베이터수', '장애인화장실수', '휠체어접근가능화장실수',
    '휠체어리프트수', '운행휠체어리프트수', '휠체어급속충전기수', '안전발판수'
]
detail_columns = [
    '엘리베이터연결층', '엘리베이터설치위치',
    '장애인화장실설치위치', '장애인화장실층',
    '휠체어리프트운행구간', '휠체어리프트설치위치',
    '휠체어급속충전기설치위치', '휠체어급속충전기층'
]
for column in count_columns:
    accessibility_station[column] = accessibility_station[column].fillna(0).astype('int64')
for column in detail_columns:
    accessibility_station[column] = accessibility_station[column].fillna('없음').replace('', '없음')

# 보유여부 컬럼은 분석과 시각화가 쉽도록 True/False가 아니라 1/0으로 저장한다.
accessibility_station['엘리베이터보유여부'] = accessibility_station['엘리베이터수'].gt(0).astype('int64')
accessibility_station['운행엘리베이터보유여부'] = accessibility_station['운행엘리베이터수'].gt(0).astype('int64')
accessibility_station['장애인화장실보유여부'] = accessibility_station['장애인화장실수'].gt(0).astype('int64')
accessibility_station['휠체어접근가능화장실보유여부'] = accessibility_station['휠체어접근가능화장실수'].gt(0).astype('int64')
accessibility_station['휠체어리프트보유여부'] = accessibility_station['휠체어리프트수'].gt(0).astype('int64')
accessibility_station['운행휠체어리프트보유여부'] = accessibility_station['운행휠체어리프트수'].gt(0).astype('int64')
accessibility_station['휠체어급속충전기보유여부'] = accessibility_station['휠체어급속충전기수'].gt(0).astype('int64')
accessibility_station['안전발판보유여부'] = accessibility_station['안전발판수'].gt(0).astype('int64')

accessibility_station = accessibility_station.sort_values(['노선명', '역명_정규화']).reset_index(drop=True)
print(f'접근성 시설 통합 역-노선 조합 수: {len(accessibility_station):,}개')
display(accessibility_station.head(20))

## 6. 장애인 승하차 데이터와 콜택시 지역 연결

### 6-1. 노선 코드 생성과 매핑 진단

In [ ]:
ridership_for_merge = ridership_wide.copy()
ridership_for_merge['노선코드'] = ridership_for_merge['호선'].astype('Int64').astype('string')

source_stations = ridership_for_merge[['노선명', '역명', '노선코드', '역명_정규화']].drop_duplicates()
mapping_audit = source_stations.merge(
    station_region, on=['노선코드', '역명_정규화'], how='left', indicator=True
)
mapping_audit['매핑상태'] = mapping_audit['_merge'].map({'both': '매핑', 'left_only': '미매핑'})
mapping_summary = (
    mapping_audit.groupby(['노선명', '매핑상태'], observed=True).size()
    .unstack(fill_value=0).sort_index()
)

mapped_count = int(mapping_audit['지역명'].notna().sum())
unmapped_count = int(mapping_audit['지역명'].isna().sum())
print(f'노선–역 매핑: {mapped_count:,}개 / 미매핑: {unmapped_count:,}개')
display(mapping_summary)
display(mapping_audit.loc[mapping_audit['매핑상태'].eq('미매핑'), ['노선명', '역명']].head(30))

### 6-2. 콜택시 지역 지하철역과 접근성 시설 결합

In [ ]:
filtered = ridership_for_merge.merge(
    station_region, on=['노선코드', '역명_정규화'], how='inner', validate='many_to_one'
).drop(columns=['노선코드', '역월평균', '역월표준편차'])

filtered = filtered.merge(
    accessibility_station,
    on=['노선명', '역명_정규화'],
    how='left',
    validate='many_to_one',
)

access_count_columns = [
    '엘리베이터수', '운행엘리베이터수', '장애인화장실수', '휠체어접근가능화장실수',
    '휠체어리프트수', '운행휠체어리프트수', '휠체어급속충전기수', '안전발판수'
]
access_bool_columns = [
    '안전발판보유여부', '엘리베이터보유여부', '운행엘리베이터보유여부',
    '장애인화장실보유여부', '휠체어접근가능화장실보유여부',
    '휠체어리프트보유여부', '운행휠체어리프트보유여부', '휠체어급속충전기보유여부'
]
for column in access_count_columns:
    filtered[column] = filtered[column].fillna(0).astype('int64')
for column in access_bool_columns:
    filtered[column] = filtered[column].fillna(0).astype('int64')

access_detail_columns = [
    '엘리베이터연결층', '엘리베이터설치위치',
    '장애인화장실설치위치', '장애인화장실층',
    '휠체어리프트운행구간', '휠체어리프트설치위치',
    '휠체어급속충전기설치위치', '휠체어급속충전기층'
]
for column in access_detail_columns:
    filtered[column] = filtered[column].fillna('없음').replace('', '없음')

# 이상치 여부도 최종 CSV에서는 1/0으로 저장한다.
filtered['이상치여부'] = filtered['이상치여부'].astype('int64')

filtered = filtered.drop(columns=['역명_정규화'])
filtered['노선정렬순서'] = filtered['호선'].astype('int64')
filtered = (
    filtered.sort_values(['사용월', '노선정렬순서', '역명', '지역명', '행정동명'])
    .drop(columns='노선정렬순서')
    .reset_index(drop=True)
)

filtered.head()

### 6-3. 필터 결과 검증

In [ ]:
assert len(filtered) > 0, '콜택시 지역과 매칭된 장애인 승하차 데이터가 없습니다.'
assert filtered['지역명'].isin(used_regions).all(), '콜택시 출발·목적 지역 외 데이터가 있습니다.'

filtered_region_dong_keys = set(zip(
    filtered['지역명'].map(normalize_area_name),
    filtered['행정동명'].map(normalize_area_name),
))
assert filtered_region_dong_keys.issubset(used_region_dong_keys), '콜택시 출발·목적 구·동 외 데이터가 있습니다.'
assert not filtered.duplicated(['사용월', '노선명', '역명']).any(), '필터 결과에 월·노선·역 중복이 있습니다.'

final_columns = [
    '사용월', '수송연월', '노선명', '호선', '고유역번호(외부역코드)', '역명',
    '지역명', '행정동명', '장애인승차인원수', '장애인하차인원수',
    '장애인총승하차인원수', '이상치여부',
    '엘리베이터수', '운행엘리베이터수', '엘리베이터보유여부', '운행엘리베이터보유여부',
    '장애인화장실수', '휠체어접근가능화장실수', '장애인화장실보유여부', '휠체어접근가능화장실보유여부',
    '휠체어리프트수', '운행휠체어리프트수', '휠체어리프트보유여부', '운행휠체어리프트보유여부',
    '휠체어급속충전기수', '휠체어급속충전기보유여부', '안전발판수', '안전발판보유여부',
    '엘리베이터연결층', '엘리베이터설치위치',
    '장애인화장실설치위치', '장애인화장실층',
    '휠체어리프트운행구간', '휠체어리프트설치위치',
    '휠체어급속충전기설치위치', '휠체어급속충전기층'
]
assert filtered[final_columns].notna().all().all(), '필터 결과 필수 컬럼에 결측값이 있습니다.'

print(f'지역 매칭 행 수: {len(filtered):,}행')
print(f'지역 매칭 역 수: {filtered["역명"].nunique():,}개')
print(f'지역 매칭 구·동 수: {filtered[["지역명", "행정동명"]].drop_duplicates().shape[0]:,}개')
print(f'운행 엘리베이터 보유 역 수: {filtered.loc[filtered["운행엘리베이터보유여부"].eq(1), ["노선명", "역명"]].drop_duplicates().shape[0]:,}개')
print(f'휠체어 접근 가능 화장실 보유 역 수: {filtered.loc[filtered["휠체어접근가능화장실보유여부"].eq(1), ["노선명", "역명"]].drop_duplicates().shape[0]:,}개')
display(filtered[final_columns].head(20))

## 7. 저장

### 7-1. 최종 컬럼 정리와 저장

In [ ]:
final_data = filtered[final_columns].copy()

# 저장 직전 최종 타입 보정: CSV에서 True/False가 아니라 1/0으로 보이도록 고정한다.
indicator_columns = [column for column in final_data.columns if column.endswith('여부')]
for column in indicator_columns:
    final_data[column] = final_data[column].replace({True: 1, False: 0, 'True': 1, 'False': 0}).astype('int64')

# 사용월은 원본이 월별 데이터라 날짜형으로 저장하면 매월 1일처럼 보인다.
# 현재 원본에는 일별 정보가 없으므로, 1일 데이터로 오해하지 않도록 YYYY-MM 문자열로 저장한다.
final_data['사용월'] = pd.to_datetime(final_data['사용월']).dt.strftime('%Y-%m')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
final_data.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

elapsed = time.perf_counter() - started_at
print(f'지하철 관련 정제 데이터 저장 완료: {OUTPUT_PATH}')
print(f'최종 행 수: {len(final_data):,}행')
print(f'필터 파일 크기: {OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB')
print(f'전체 실행 시간: {elapsed:.2f}초')
print(f'0/1 변환 컬럼: {indicator_columns}')
display(final_data.head())